# Unit 3: CNN核心组件与模型构建

## 学习目标
- 掌握PyTorch中CNN核心层的实现
- 学会使用nn.Module构建自定义模型
- 理解不同网络层的作用与参数配置
- 掌握Sequential与函数式API的使用
- 学会查看和管理模型参数

## 参考资源
- [PyTorch官方文档 - nn.Module](https://pytorch.org/docs/stable/generated/torch.nn.Module.html)
- [PyTorch官方文档 - Convolution Layers](https://pytorch.org/docs/stable/nn.html#convolution-layers)
- [PyTorch教程 - Learn the Basics](https://pytorch.org/tutorials/beginner/basics/intro.html)

## 3.1 卷积层(nn.Conv2d)详解

nn.Conv2d是PyTorch中最常用的2D卷积层。

### 主要参数
- **in_channels**: 输入通道数
- **out_channels**: 输出通道数(卷积核数量)
- **kernel_size**: 卷积核大小
- **stride**: 步长(默认1)
- **padding**: 填充(默认0)
- **dilation**: 空洞卷积的膨胀率(默认1)
- **groups**: 分组卷积的组数(默认1)
- **bias**: 是否使用偏置(默认True)

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

print("=" * 60)
print("3.1.1 基础卷积层使用")
print("=" * 60)

conv1 = nn.Conv2d(in_channels=3, out_channels=16, kernel_size=3, padding=1)
print(f"卷积层: {conv1}")
print(f"权重形状: {conv1.weight.shape}")
print(f"偏置形状: {conv1.bias.shape}")

x = torch.randn(1, 3, 32, 32)
output = conv1(x)
print(f"\n输入形状: {x.shape}")
print(f"输出形状: {output.shape}")

In [ ]:
print("=" * 60)
print("3.1.2 不同参数配置的效果")
print("=" * 60)

x = torch.randn(1, 3, 64, 64)
print(f"输入形状: {x.shape}\n")

conv_configs = [
    {"in_channels": 3, "out_channels": 16, "kernel_size": 3, "padding": 1},
    {"in_channels": 3, "out_channels": 16, "kernel_size": 5, "padding": 2},
    {"in_channels": 3, "out_channels": 16, "kernel_size": 3, "stride": 2, "padding": 1},
    {"in_channels": 3, "out_channels": 16, "kernel_size": 3, "dilation": 2, "padding": 2},
]

for i, config in enumerate(conv_configs, 1):
    conv = nn.Conv2d(**config)
    out = conv(x)
    print(f"配置{i}: {config}")
    print(f"  输出形状: {out.shape}\n")

## 3.2 池化层

池化层用于降低特征图的空间尺寸。

In [ ]:
print("=" * 60)
print("3.2 池化层演示")
print("=" * 60)

x = torch.randn(1, 16, 32, 32)
print(f"输入形状: {x.shape}\n")

max_pool = nn.MaxPool2d(kernel_size=2, stride=2)
out_max = max_pool(x)
print(f"最大池化(2x2, stride=2): {out_max.shape}")

avg_pool = nn.AvgPool2d(kernel_size=2, stride=2)
out_avg = avg_pool(x)
print(f"平均池化(2x2, stride=2): {out_avg.shape}")

adaptive_pool = nn.AdaptiveAvgPool2d((1, 1))
out_adaptive = adaptive_pool(x)
print(f"自适应平均池化(1x1): {out_adaptive.shape}")

## 3.3 批归一化(Batch Normalization)

批归一化通过对每个mini-batch进行归一化，加速训练并提高模型稳定性。

In [ ]:
print("=" * 60)
print("3.3 批归一化演示")
print("=" * 60)

x = torch.randn(8, 16, 32, 32)
print(f"输入形状: {x.shape}")
print(f"输入均值: {x.mean():.4f}, 标准差: {x.std():.4f}")

bn = nn.BatchNorm2d(16)
out_bn = bn(x)
print(f"\n批归一化后形状: {out_bn.shape}")
print(f"输出均值: {out_bn.mean():.4f}, 标准差: {out_bn.std():.4f}")

print(f"\nBatchNorm参数:")
print(f"  gamma(权重)形状: {bn.weight.shape}")
print(f"  beta(偏置)形状: {bn.bias.shape}")
print(f"  running_mean形状: {bn.running_mean.shape}")
print(f"  running_var形状: {bn.running_var.shape}")

## 3.4 Dropout层

Dropout通过随机丢弃神经元来防止过拟合。

In [ ]:
print("=" * 60)
print("3.4 Dropout演示")
print("=" * 60)

x = torch.ones(10)
dropout = nn.Dropout(p=0.5)

print(f"输入: {x}")

dropout.train()
out_train = dropout(x)
print(f"训练模式下: {out_train}")
print(f"被丢弃的比例: {(out_train == 0).sum().item() / len(x) * 100:.0f}%")

dropout.eval()
out_eval = dropout(x)
print(f"\n评估模式下: {out_eval}")
print(f"评估模式下不会丢弃神经元")

## 3.5 使用nn.Module构建自定义模型

所有PyTorch模型都应继承自nn.Module类。

In [ ]:
print("=" * 60)
print("3.5.1 定义自定义CNN模型")
print("=" * 60)

class CustomCNN(nn.Module):
    def __init__(self, num_classes=10):
        super(CustomCNN, self).__init__()
        
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(64 * 8 * 8, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )
    
    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

model = CustomCNN(num_classes=10)
print(model)

In [ ]:
print("=" * 60)
print("3.5.2 模型参数统计")
print("=" * 60)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"总参数量: {total_params:,}")
print(f"可训练参数量: {trainable_params:,}")
print(f"\n各层参数:")
for name, param in model.named_parameters():
    print(f"  {name}: {param.shape}, requires_grad={param.requires_grad}")

In [ ]:
print("=" * 60)
print("3.5.3 模型前向传播测试")
print("=" * 60)

dummy_input = torch.randn(4, 3, 32, 32)
output = model(dummy_input)

print(f"输入形状: {dummy_input.shape}")
print(f"输出形状: {output.shape}")
print(f"输出值范围: [{output.min():.4f}, {output.max():.4f}]")

## 3.6 Sequential与函数式API对比

In [ ]:
print("=" * 60)
print("3.6 Sequential vs 函数式API")
print("=" * 60)

class SequentialModel(nn.Module):
    def __init__(self):
        super(SequentialModel, self).__init__()
        self.net = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
        )
    
    def forward(self, x):
        return self.net(x)

class FunctionalModel(nn.Module):
    def __init__(self):
        super(FunctionalModel, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
    
    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        return x

x = torch.randn(1, 3, 32, 32)

model_seq = SequentialModel()
model_func = FunctionalModel()

print(f"Sequential模型输出形状: {model_seq(x).shape}")
print(f"Functional模型输出形状: {model_func(x).shape}")

## 3.7 多分支网络结构

In [ ]:
print("=" * 60)
print("3.7 多分支网络(Inception风格)")
print("=" * 60)

class MultiBranchNet(nn.Module):
    def __init__(self, in_channels, out_channels):
        super(MultiBranchNet, self).__init__()
        
        self.branch1 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
        )
        
        self.branch2 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
        )
        
        self.branch3 = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=5, padding=2),
            nn.ReLU(inplace=True),
        )
    
    def forward(self, x):
        out1 = self.branch1(x)
        out2 = self.branch2(x)
        out3 = self.branch3(x)
        
        output = torch.cat([out1, out2, out3], dim=1)
        return output

multi_branch = MultiBranchNet(in_channels=64, out_channels=32)
x = torch.randn(2, 64, 16, 16)
output = multi_branch(x)

print(f"输入形状: {x.shape}")
print(f"输出形状: {output.shape}")
print(f"输出通道数 = 3 * 32 = 96 (三个分支拼接)")

## 3.8 模型设备管理

In [ ]:
print("=" * 60)
print("3.8 模型设备管理")
print("=" * 60)

from utils import get_device

device = get_device()
print(f"可用设备: {device}")

model = CustomCNN(num_classes=10).to(device)
print(f"模型已移动到: {next(model.parameters()).device}")

x = torch.randn(2, 3, 32, 32).to(device)
output = model(x)
print(f"输出设备: {output.device}")

## 本章小结

本单元我们学习了：
1. 卷积层(nn.Conv2d)的详细参数配置
2. 池化层(最大池化、平均池化、自适应池化)
3. 批归一化(BatchNorm2d)的原理与使用
4. Dropout层的作用与训练/评估模式的区别
5. 使用nn.Module构建自定义模型
6. Sequential与函数式API的对比
7. 多分支网络结构的设计
8. 模型参数统计与设备管理

## 练习建议
1. 尝试构建不同深度的CNN模型
2. 比较有无BatchNorm对模型训练的影响
3. 实验不同Dropout比率的效果
4. 设计一个包含多分支结构的网络

## 下一步
进入Unit 4，学习完整的模型训练流程与优化方法。